# Module 3, Part 2 — From LeNet to ResNet

This notebook is the runnable companion to Part 2 of the Module 03 learning
module. It reproduces, in code, the central claim of that section:

> Simply stacking more plain convolutional layers past a certain depth
> makes training *worse*, not better, because of vanishing gradients, and a
> residual (skip) connection fixes this by guaranteeing the gradient a
> direct path back to earlier layers.

We build a small, deliberately plain, deep convolutional stack, show its
gradients vanishing as depth increases, then rebuild the same depth with
residual connections and show the gradient surviving. Everything here runs
on CPU in well under a minute.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


## 1. The standard block, stacked plain

A "plain" CNN block here is: `Conv2d -> ReLU`, keeping channels and
spatial size fixed (`padding=1`, `stride=1`) so we can stack as many as we
like without the input shrinking to nothing. This mirrors the module's
description of the classic LeNet/VGGNet recipe, just without the
pooling/downsampling steps, so depth is the only variable we are changing.

**A deliberate choice:** this first block leaves out `BatchNorm2d`, even
though the module's own `ResidualBlock` code example includes it. Section
2 shows why: BatchNorm's own normalization interacts with backpropagation
in ways that change the picture, and the cleanest way to see the plain
vanishing-gradient story from the module is without it. Section 4 goes
back to the module's exact code (with BatchNorm included) to check its
specific algebraic claim about the skip connection.

In [2]:
class PlainConvBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        return torch.relu(self.conv(x))


class PlainConvStack(nn.Module):
    """`depth` plain blocks in a row, same channel count throughout."""

    def __init__(self, depth, channels=16):
        super().__init__()
        self.blocks = nn.ModuleList([PlainConvBlock(channels) for _ in range(depth)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


# Quick shape check: depth shouldn't change height/width or channel count.
test_stack = PlainConvStack(depth=5, channels=16)
test_x = torch.randn(2, 16, 8, 8)
test_y = test_stack(test_x)
print("Input shape: ", tuple(test_x.shape))
print("Output shape:", tuple(test_y.shape))
assert test_y.shape == test_x.shape

Input shape:  (2, 16, 8, 8)
Output shape: (2, 16, 8, 8)


## 2. Watching the gradient vanish as depth increases

For a range of depths, we run one forward pass, take the mean of the
output as a stand-in "loss", call `.backward()`, and record the gradient
norm at the very first convolutional layer. If gradients are vanishing,
that first-layer gradient should shrink toward zero as depth grows, even
though nothing else about the architecture changed.

(We tried this first *with* BatchNorm in each block, matching the
module's `ResidualBlock` code exactly. The honest result: the gradient
norm grew with depth instead of shrinking, because BatchNorm's own
normalization reshapes how gradients scale through a deep stack. That is
a real and interesting effect, but it is not the simple vanishing-gradient
story the module is illustrating here, so the comparison below uses the
plain `Conv2d -> ReLU` block from Section 1, with no normalization layer
to complicate the picture.)

In [3]:
def first_layer_grad_norm(depth, channels=16, size=8, seed=0):
    torch.manual_seed(seed)
    stack = PlainConvStack(depth=depth, channels=channels)
    x = torch.randn(4, channels, size, size)

    output = stack(x)
    loss = output.mean()
    loss.backward()

    first_conv = stack.blocks[0].conv
    return first_conv.weight.grad.norm().item()


depths = [1, 2, 4, 8, 16, 32, 64]
results = {}
print(f"{'depth':>6} | {'first-layer grad norm':>22}")
print("-" * 32)
for d in depths:
    grad_norm = first_layer_grad_norm(d)
    results[d] = grad_norm
    print(f"{d:>6} | {grad_norm:>22.10f}")

 depth |  first-layer grad norm
--------------------------------
     1 |           0.1456413865
     2 |           0.0669282451
     4 |           0.0075164246
     8 |           0.0001247421
    16 |           0.0000000225
    32 |           0.0000000000
    64 |           0.0000000000


In [4]:
shallow, deep = depths[0], depths[-1]
print(f"depth-{shallow} gradient norm: {results[shallow]:.10f}")
print(f"depth-{deep} gradient norm: {results[deep]:.10f}")
if results[deep] > 0:
    print(f"The depth-{shallow} gradient norm is {results[shallow] / results[deep]:,.0f}x larger.")
else:
    print(f"The depth-{deep} gradient has underflowed to exactly zero in float32:")
    print("not just small, but numerically indistinguishable from no gradient at all.")

print("\nThis is the module's claim made concrete: nothing about the architecture")
print("changed except depth, and the gradient reaching the first layer collapsed")
print("by many orders of magnitude, exactly the same chain-rule mechanism as the")
print("sigmoid network in Module 02, now with convolutional layers instead of")
print("linear ones.")

depth-1 gradient norm: 0.1456413865
depth-64 gradient norm: 0.0000000000
The depth-64 gradient has underflowed to exactly zero in float32:
not just small, but numerically indistinguishable from no gradient at all.

This is the module's claim made concrete: nothing about the architecture
changed except depth, and the gradient reaching the first layer collapsed
by many orders of magnitude, exactly the same chain-rule mechanism as the
sigmoid network in Module 02, now with convolutional layers instead of
linear ones.


## 3. The residual block fixes it, with nothing else changed

Now the same `PlainConvBlock`, but each one adds its input back to its
output: `y = relu(conv(x) + x)`. Depth, channel count, and the random seed
all stay identical to Section 2, so any difference in the result is
attributable to the skip connection alone.

In [5]:
class ResidualConvBlock(nn.Module):
    """Same conv layer as PlainConvBlock, but with a skip connection."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        return torch.relu(self.conv(x) + x)   # the skip connection


class ResidualConvStack(nn.Module):
    def __init__(self, depth, channels=16):
        super().__init__()
        self.blocks = nn.ModuleList([ResidualConvBlock(channels) for _ in range(depth)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


def first_layer_grad_norm_residual(depth, channels=16, size=8, seed=0):
    torch.manual_seed(seed)
    stack = ResidualConvStack(depth=depth, channels=channels)
    x = torch.randn(4, channels, size, size)

    output = stack(x)
    loss = output.mean()
    loss.backward()

    first_conv = stack.blocks[0].conv
    return first_conv.weight.grad.norm().item()

In [6]:
print(f"{'depth':>6} | {'plain stack':>16} | {'residual stack':>16}")
print("-" * 44)
residual_results = {}
for d in depths:
    plain_g = results[d]
    res_g = first_layer_grad_norm_residual(d)
    residual_results[d] = res_g
    print(f"{d:>6} | {plain_g:>16.10f} | {res_g:>16.10f}")

print(f"\nAt depth {deep}, the plain stack's gradient has vanished to {results[deep]:.1f},")
print(f"while the identical stack with skip connections still has a gradient norm")
print(f"of {residual_results[deep]:.2f}. Same depth, same per-layer computation,")
print("the only difference is the \"+ x\" on each block, and that alone is the")
print("difference between a dead first layer and a healthy one.")

 depth |      plain stack |   residual stack
--------------------------------------------
     1 |     0.1456413865 |     0.1486096531
     2 |     0.0669282451 |     0.1805054694
     4 |     0.0075164246 |     0.2747628093


     8 |     0.0001247421 |     0.4666946232
    16 |     0.0000000225 |     0.9388743043
    32 |     0.0000000000 |     4.5354700089
    64 |     0.0000000000 |   275.9313354492

At depth 64, the plain stack's gradient has vanished to 0.0,
while the identical stack with skip connections still has a gradient norm
of 275.93. Same depth, same per-layer computation,
the only difference is the "+ x" on each block, and that alone is the
difference between a dead first layer and a healthy one.


## 4. Confirming the "+1" directly, on the module's exact code

Back to the module's own `ResidualBlock`, with `BatchNorm2d` included
exactly as written there: `conv1 -> bn1 -> relu -> conv2 -> bn2 -> + identity -> relu`.
This section is not about depth or vanishing gradients; it is a single,
narrow algebraic check of the skip connection itself, which holds
regardless of what else is inside `F(x)`.

The claim is about the pre-activation sum `z = F(x) + x`, before any
further nonlinearity: `dz/dx = dF(x)/dx + 1`. (`ResidualBlock.forward`
applies one more ReLU *after* that addition, which is not part of the
claim, so this check works directly with `z` rather than the block's
final output.) We compute `z` and `F(x)` by hand from the same layers the
block uses, then compare their gradients.

In [7]:
class ResidualBlock(nn.Module):
    """The module's own block, BatchNorm included, exactly as printed there."""

    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity          # the skip connection
        return torch.relu(out)


torch.manual_seed(0)
channels = 4
block = ResidualBlock(channels)
block.eval()  # fix BatchNorm statistics so the block is a deterministic function of x

x = torch.randn(1, channels, 4, 4, requires_grad=True)

# F(x): everything in the block up to (but not including) the skip addition.
F_x = block.bn2(block.conv2(torch.relu(block.bn1(block.conv1(x)))))

# z: the pre-activation sum, F(x) + x, i.e. the block's internal residual
# addition before its own trailing ReLU.
z = F_x + x

grad_F_only = torch.autograd.grad(F_x.sum(), x, retain_graph=True)[0]
grad_z = torch.autograd.grad(z.sum(), x, retain_graph=True)[0]

difference = grad_z - grad_F_only
print("grad of F(x) alone, mean:         ", grad_F_only.mean().item())
print("grad of z = F(x) + x, mean:       ", grad_z.mean().item())
print("difference, every element (should all be 1.0):")
print(difference)
assert torch.allclose(difference, torch.ones_like(difference), atol=1e-5)
print("\nConfirmed: dz/dx = dF(x)/dx + 1, exactly, at every element.")
print("The skip connection adds exactly one unimpeded unit of gradient,")
print("regardless of how small dF(x)/dx becomes.")

grad of F(x) alone, mean:          0.02976253814995289
grad of z = F(x) + x, mean:        1.029762625694275
difference, every element (should all be 1.0):


tensor([[[[1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000]],

         [[1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000]],

         [[1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000]],

         [[1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000],
          [1.0000, 1.0000, 1.0000, 1.0000]]]])

Confirmed: dz/dx = dF(x)/dx + 1, exactly, at every element.
The skip connection adds exactly one unimpeded unit of gradient,
regardless of how small dF(x)/dx becomes.


## Summary

- A plain stack of `Conv2d -> ReLU` blocks, with nothing else changed,
  shows a first-layer gradient that collapses by many orders of magnitude
  as depth grows from 1 to 64, underflowing to exactly zero in float32.
  This is the same chain-rule mechanism from Module 02, now at CNN scale.
- Rebuilding the identical stack with residual (skip) connections keeps
  the first-layer gradient alive and orders of magnitude larger at the
  same depth, with no other change to the architecture.
- Adding `BatchNorm2d` back in, matching the module's own `ResidualBlock`
  code exactly, changes this picture: the normalization itself reshapes
  how gradients scale through the stack, and a careful look at a real
  architecture is never as simple as one slide's worth of story. That is
  a worthwhile lesson on its own, separate from the core vanishing-gradient
  mechanism this notebook isolates.
- The algebraic claim `dy/dx = dF(x)/dx + 1` for `y = F(x) + x` checks out
  exactly under `torch.autograd` on the module's exact `ResidualBlock`,
  not just as a hand-wave.

This is also the direct code sequel to Lab 02's dying-ReLU diagnosis: same
underlying failure mode (a gradient that cannot get back to early layers),
one architectural family later, with the architecture itself as the fix
instead of a change in initialization or normalization alone.